# Atividade A12 - Analise de Dados: Wine Quality

**Disciplina:** Metodos Quantitativos  
**Base:** Wine Quality, UCI Machine Learning Repository  
**Objetivo:** responder ao exercicio em `Trabalho-A12-MetodosQuantitativos-CC-2026.pdf` usando a base escolhida no repositorio.

Este caderno calcula, para cada variavel de entrada e para a variavel que se deseja prever (`quality`):

- media, mediana e moda;
- justificativa quando alguma medida nao pode ser calculada;
- classificacao da distribuicao como simetrica, assimetrica a direita ou assimetrica a esquerda;
- desvio absoluto medio, variancia e desvio padrao;
- percentual de observacoes alem de um desvio padrao da media.

## 1. Contexto do repositorio e da base

O repositorio trabalha com o dataset **Wine Quality**, composto por amostras de vinhos portugueses do tipo *Vinho Verde*:

- `sample_data/winequality-red.csv`: vinhos tintos;
- `sample_data/winequality-white.csv`: vinhos brancos;
- `quality`: nota sensorial ordinal, usada como variavel alvo;
- `wine_type`: variavel nominal criada neste projeto para identificar a origem da amostra.

As demais variaveis sao medicoes fisico-quimicas usadas como variaveis de entrada.

In [29]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib_cache"))
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    def display(*objects: object, **kwargs: object) -> None:
        for obj in objects:
            print(obj)

plt.rcParams.update({
    "axes.grid": True,
    "grid.alpha": 0.25,
    "figure.facecolor": "white",
    "axes.facecolor": "white"
})
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.4f}".format)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "sample_data"
EXPORT_DIR = BASE_DIR / "export"
EXPORT_DIR.mkdir(exist_ok=True)

## 2. Carregamento e validacao dos dados

Os dois arquivos locais sao carregados e combinados. A coluna `wine_type` e adicionada para preservar a identificacao entre vinho tinto e vinho branco.

In [30]:
red = pd.read_csv(DATA_DIR / "winequality-red.csv", sep=";")
white = pd.read_csv(DATA_DIR / "winequality-white.csv", sep=";")

red["wine_type"] = "red"
white["wine_type"] = "white"

df = pd.concat([red, white], ignore_index=True)

print("Dimensao da base combinada:", df.shape)
display(df.head())
display(df.isna().sum().rename("valores_ausentes"))
display(df["wine_type"].value_counts().rename("quantidade"))

## 3. Classificacao das variaveis

Para responder ao exercicio, as variaveis foram separadas por tipo estatistico:

- **continuas:** medicoes fisico-quimicas;
- **ordinal:** `quality`, nota sensorial ordenada e variavel alvo;
- **nominal:** `wine_type`, identificacao de tipo de vinho.

Medidas como media, variancia e desvio padrao so sao interpretaveis para variaveis numericas. Para `wine_type`, apenas a moda e frequencias fazem sentido.

In [31]:
continuous_columns = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar",
    "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density",
    "pH", "sulphates", "alcohol"
]
target_column = "quality"
nominal_columns = ["wine_type"]

variable_roles = pd.DataFrame({
    "variavel": continuous_columns + [target_column] + nominal_columns,
    "papel": ["entrada"] * len(continuous_columns) + ["alvo"] + ["identificacao"],
    "tipo": ["continua"] * len(continuous_columns) + ["ordinal"] + ["nominal"]
})

display(variable_roles)

## 4. Criterios de calculo

Criterios usados no caderno:

- **Desvio absoluto medio:** media de `|x - media|`.
- **Variancia e desvio padrao:** medidas amostrais, com `ddof=1`.
- **Percentual alem de 1 desvio padrao:** percentual de observacoes com valor menor que `media - desvio_padrao` ou maior que `media + desvio_padrao`.
- **Assimetria:** definida pela comparacao entre media, mediana e moda principal. Quando media, mediana e moda ficam proximas, a distribuicao e tratada como aproximadamente simetrica. Quando a media fica acima da mediana/moda, indica cauda a direita; quando fica abaixo, indica cauda a esquerda.

In [32]:
def format_modes(series: pd.Series, max_items: int = 6) -> str:
    modes = series.mode(dropna=True).tolist()
    if not modes:
        return "Nao ha moda"
    shown = modes[:max_items]
    formatted = ", ".join(f"{x:.4f}" if isinstance(x, (float, np.floating)) else str(x) for x in shown)
    if len(modes) > max_items:
        formatted += f" ... ({len(modes)} modas)"
    return formatted


def primary_mode(series: pd.Series):
    modes = series.mode(dropna=True)
    if modes.empty:
        return np.nan
    return modes.iloc[0]


def classify_numeric_distribution(series: pd.Series) -> tuple[str, str]:
    clean = series.dropna()
    mean = clean.mean()
    median = clean.median()
    mode = primary_mode(clean)
    std = clean.std(ddof=1)

    if pd.isna(mode) or std == 0 or pd.isna(std):
        return "Nao classificavel", "Nao ha variacao suficiente ou nao ha moda definida."

    tolerance = 0.05 * std
    close_mean_median = abs(mean - median) <= tolerance
    close_median_mode = abs(median - mode) <= tolerance

    if close_mean_median and close_median_mode:
        return (
            "Aproximadamente simetrica",
            "Media, mediana e moda principal estao proximas."
        )
    if mean > median:
        return (
            "Assimetrica a direita",
            "A media ficou maior que a mediana; isso indica cauda mais longa a direita."
        )
    if mean < median:
        return (
            "Assimetrica a esquerda",
            "A media ficou menor que a mediana; isso indica cauda mais longa a esquerda."
        )
    if median > mode:
        return (
            "Assimetrica a direita",
            "A mediana ficou maior que a moda principal, sugerindo deslocamento para valores maiores."
        )
    return (
        "Assimetrica a esquerda",
        "A mediana ficou menor que a moda principal, sugerindo deslocamento para valores menores."
    )


def summarize_variable(df: pd.DataFrame, column: str, var_type: str, role: str) -> dict:
    series = df[column]
    result = {
        "variavel": column,
        "papel": role,
        "tipo": var_type,
        "n": int(series.notna().sum()),
        "media": np.nan,
        "mediana": np.nan,
        "moda": format_modes(series),
        "desvio_absoluto_medio": np.nan,
        "variancia_amostral": np.nan,
        "desvio_padrao_amostral": np.nan,
        "percentual_alem_1_dp": np.nan,
        "distribuicao": "Nao aplicavel",
        "justificativa": "",
        "medidas_nao_calculaveis": ""
    }

    if pd.api.types.is_numeric_dtype(series):
        clean = series.dropna()
        mean = clean.mean()
        median = clean.median()
        std = clean.std(ddof=1)
        lower = mean - std
        upper = mean + std

        distribution, justification = classify_numeric_distribution(clean)
        result.update({
            "media": mean,
            "mediana": median,
            "desvio_absoluto_medio": (clean - mean).abs().mean(),
            "variancia_amostral": clean.var(ddof=1),
            "desvio_padrao_amostral": std,
            "percentual_alem_1_dp": ((clean < lower) | (clean > upper)).mean() * 100,
            "distribuicao": distribution,
            "justificativa": justification,
            "medidas_nao_calculaveis": "Nenhuma para fins descritivos numericos."
        })
    else:
        result.update({
            "justificativa": "Variavel nominal; nao possui escala numerica nem ordem natural.",
            "medidas_nao_calculaveis": "Media, mediana, desvio absoluto medio, variancia, desvio padrao e percentual alem de 1 desvio padrao nao se aplicam."
        })
    return result

summary_rows = [
    summarize_variable(df, row.variavel, row.tipo, row.papel)
    for row in variable_roles.itertuples(index=False)
]
summary = pd.DataFrame(summary_rows)

numeric_cols = [
    "media", "mediana", "desvio_absoluto_medio", "variancia_amostral",
    "desvio_padrao_amostral", "percentual_alem_1_dp"
]
summary_display = summary.copy()
summary_display[numeric_cols] = summary_display[numeric_cols].round(4)

display(summary_display)

## 5. Frequencias da variavel alvo e da variavel nominal

Como `quality` e ordinal e `wine_type` e nominal, as frequencias ajudam a interpretar a distribuicao de forma mais adequada.

In [33]:
quality_freq = (
    df["quality"]
    .value_counts()
    .sort_index()
    .rename_axis("quality")
    .reset_index(name="frequencia")
)
quality_freq["percentual"] = quality_freq["frequencia"] / len(df) * 100

wine_type_freq = (
    df["wine_type"]
    .value_counts()
    .rename_axis("wine_type")
    .reset_index(name="frequencia")
)
wine_type_freq["percentual"] = wine_type_freq["frequencia"] / len(df) * 100

display(quality_freq)
display(wine_type_freq)

## 6. Visualizacoes de apoio

Os graficos abaixo ajudam a conferir visualmente as conclusoes sobre assimetria e dispersao. As linhas verticais indicam media e mediana.

As imagens tambem sao salvas em `export/a12_graficos/` para facilitar a entrega e revisao.

In [34]:
GRAPH_DIR = EXPORT_DIR / "a12_graficos"
GRAPH_DIR.mkdir(exist_ok=True)

plot_columns = continuous_columns + [target_column]
fig, axes = plt.subplots(4, 3, figsize=(18, 16))
axes = axes.ravel()

for ax, col in zip(axes, plot_columns):
    values = df[col].dropna()
    ax.hist(values, bins="auto", color="#2f6f73", edgecolor="white", alpha=0.85)
    ax.axvline(values.mean(), color="#b83232", linestyle="--", linewidth=1.5, label="media")
    ax.axvline(values.median(), color="#1f3a93", linestyle=":", linewidth=1.8, label="mediana")
    ax.set_title(col)
    ax.legend(fontsize=8)

for ax in axes[len(plot_columns):]:
    ax.axis("off")

plt.tight_layout()
combined_path = GRAPH_DIR / "histogramas_variaveis_numericas.png"
fig.savefig(combined_path, dpi=160, bbox_inches="tight")
display(fig)
plt.close(fig)

saved_files = [combined_path]

for col in plot_columns:
    values = df[col].dropna()
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(values, bins="auto", color="#2f6f73", edgecolor="white", alpha=0.85)
    ax.axvline(values.mean(), color="#b83232", linestyle="--", linewidth=1.5, label="media")
    ax.axvline(values.median(), color="#1f3a93", linestyle=":", linewidth=1.8, label="mediana")
    ax.set_title(f"Distribuicao de {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Frequencia")
    ax.legend()
    fig.tight_layout()
    file_name = "hist_" + col.replace(" ", "_").replace("/", "_") + ".png"
    file_path = GRAPH_DIR / file_name
    fig.savefig(file_path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    saved_files.append(file_path)

print("Graficos salvos em:")
for file_path in saved_files:
    print(file_path)

## 7. Resposta consolidada do exercicio

A tabela a seguir e a resposta principal da atividade. Ela apresenta, para cada variavel, as medidas solicitadas e a justificativa de simetria/assimetria.

In [35]:
answer_columns = [
    "variavel", "papel", "tipo", "media", "mediana", "moda",
    "distribuicao", "justificativa", "desvio_absoluto_medio",
    "variancia_amostral", "desvio_padrao_amostral", "percentual_alem_1_dp",
    "medidas_nao_calculaveis"
]
answer = summary_display[answer_columns]
display(answer)

output_path = EXPORT_DIR / "a12_resumo_medidas_wine_quality.csv"
summary.to_csv(output_path, index=False)
print("Tabela consolidada salva em:", output_path)

## 8. Conclusoes

Com base nos calculos:

- As variaveis fisico-quimicas sao numericas e permitem calcular todas as medidas solicitadas.
- A variavel `quality`, apesar de ordinal, foi tratada numericamente para fins descritivos porque suas categorias sao notas ordenadas de 0 a 10.
- A variavel `wine_type` e nominal; por isso, media, mediana, dispersao e percentual alem de um desvio padrao nao possuem interpretacao estatistica adequada.
- A maioria das variaveis fisico-quimicas apresenta assimetria, o que e esperado em dados reais de composicao quimica e controle de qualidade.
- A variavel alvo `quality` concentra a maior parte das observacoes nas notas 5 e 6, mostrando uma base desbalanceada em relacao a notas extremas.